# 🎨 Hello Clova Agent — 발표 슬라이드 자동 생성 체험

**한국어 프롬프트 한 줄 → Reveal.js 발표 슬라이드 자동 생성**

---

## 이 노트북의 목적

Gamma, SkyAI 같은 발표 자동 생성 서비스의 **핵심 파이프라인을 직접 구현하고 체험**합니다.

체험 과정에서 자연스럽게 아래 개념을 익힐 수 있습니다:

| 개념 | 이 프로젝트에서 | 코드 위치 |
|------|----------------|----------|
| **API 서버** | Ollama가 LLM을 HTTP API로 제공 | 셀 2~3 |
| **앱 서버** | LangGraph 4-노드 에이전트 파이프라인 | `agent/graph.py` |
| **웹 서버** | Gradio가 브라우저 요청을 처리 | `ui/app.py` |

---

## 실행 방법

**셀을 위에서 아래로 순서대로 실행하세요.** (단축키: `Shift + Enter`)

> ⏱️ 처음 실행 시 모델 다운로드로 약 5~10분 소요됩니다.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 1/6  ▌ Ollama 설치
#
# Ollama: LLM을 로컬에서 손쉽게 실행하는 프레임워크
#
# [Colab 문법 안내]
#   !  →  터미널(bash) 명령어 실행
#   %  →  Jupyter 전용 명령어 (매직 커맨드)
# ══════════════════════════════════════════════════════════════════════

print("📦 Ollama 설치 중... (약 30초 소요)")

!sudo apt-get update -qq && sudo apt-get install -y zstd curl -qq
!curl -fsSL https://ollama.com/install.sh | sh

print("\n✅ Ollama 설치 완료")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 2/6  ▌ Ollama API 서버 시작
#
# [개념] API 서버(API Server)란?
#   LLM 모델을 HTTP 엔드포인트로 제공하는 서버입니다.
#   이 서버가 실행되면 다른 프로그램이 LLM을 마치
#   외부 서비스처럼 호출할 수 있습니다.
#
#   엔드포인트: http://localhost:11434/v1/chat/completions
#
# subprocess.Popen: 프로그램을 '백그라운드'에서 실행
#   → 노트북 셀이 완료되어도 서버는 계속 동작
# ══════════════════════════════════════════════════════════════════════

import subprocess
import time
import urllib.request

# 기존 Ollama 프로세스 종료 (중복 실행 방지)
subprocess.run(["pkill", "-9", "ollama"], capture_output=True)
time.sleep(1)

# 백그라운드로 Ollama API 서버 시작
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# 서버 준비 대기 (최대 30초)
print("⏳ Ollama API 서버 시작 대기 중", end="")
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=1)
        print("\n✅ Ollama API 서버 동작 중 — http://localhost:11434")
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(1)
else:
    print("\n❌ 서버 시작 실패 — 셀을 다시 실행해 주세요.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 3/6  ▌ LLM 모델 다운로드
#
# ── 모델 선택 안내 ──────────────────────────────────────────────────
#
# [기본] Qwen 2.5 7B  (한국어 지원, ~4.7 GB, Ollama 사용)
#   MODEL_NAME = "qwen2.5:7b"
#
# [국산 LLM] HyperCLOVA X SEED Think-14B  ← 공공기관/국산 LLM 요건
#   - Ollama 공식 레지스트리 미등록 → vLLM 또는 GGUF Modelfile 필요
#   - vLLM + 4-bit 양자화 시 ~8-10GB VRAM (T4 16GB 가능)
#   - 사용법: 아래 MODEL_NAME을 변경하고 셀 2를 건너뛴 뒤
#     셀 3 이후에 있는 [HyperCLOVA X vLLM 셀]을 실행하세요
#   MODEL_NAME = "naver-hyperclovax/HyperCLOVA-X-SEED-Think-14B"
#
# HyperCLOVA X SEED VRAM 요구량:
#   ┌──────────────────────────────────┬─────────────┬────────────┐
#   │ 모델                             │ 방식        │ 필요 VRAM  │
#   ├──────────────────────────────────┼─────────────┼────────────┤
#   │ SEED-Instruct-3B                 │ fp16 vLLM   │ ~8 GB      │
#   │ SEED-Think-14B  ← T4 권장       │ 4-bit vLLM  │ ~8-10 GB   │
#   │ SEED-Think-32B                   │ 4-bit vLLM  │ ~18 GB     │
#   └──────────────────────────────────┴─────────────┴────────────┘
# ══════════════════════════════════════════════════════════════════════

import os

# ↓ 모델 변경이 필요하면 여기서 수정하세요
MODEL_NAME = "qwen2.5:7b"

os.environ["LLM_MODEL"] = MODEL_NAME   # 이후 셀에서 참조

print(f"📥 모델 다운로드 중: {MODEL_NAME}")
print("   처음 실행 시 수 분이 소요됩니다 (진행률이 표시됩니다)")
print()

!ollama pull {MODEL_NAME}

print(f"\n✅ 모델 준비 완료: {MODEL_NAME}")

---

## 🇰🇷 [선택] HyperCLOVA X SEED Think-14B — 국산 LLM 옵션

> 공공기관 납품 또는 국산 LLM 요건이 있는 경우 아래 셀을 실행하세요.  
> **셀 2(Ollama 서버)와 셀 3(모델 다운로드)은 실행하지 않아도 됩니다.**

| 항목 | 내용 |
|------|------|
| 모델 | `naver-hyperclovax/HyperCLOVA-X-SEED-Think-14B` |
| VRAM | ~8-10 GB (4-bit bitsandbytes 양자화) |
| 서빙 | vLLM (Ollama 아님 — 공식 레지스트리 미등록) |
| Colab | T4 (15GB) 또는 A100 사용 권장 |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# [선택] HyperCLOVA X SEED Think-14B — vLLM + 4-bit 양자화
#
# 이 셀 실행 전 확인사항:
#   1) 셀 4 (git clone) 및 셀 5 (의존성 설치) 는 먼저 실행하세요
#   2) Colab 런타임 → GPU: T4 (15GB) 또는 A100 선택
#   3) 셀 2 (Ollama 서버) 를 실행했다면 중지해 주세요
#
# 실행하면 vLLM 서버가 백그라운드에서 시작됩니다.
# 모델 다운로드 (~28GB bf16 → 4-bit 적재) 약 5-15분 소요.
# ══════════════════════════════════════════════════════════════════════

import subprocess, time, os, urllib.request

HCX_MODEL = "naver-hyperclovax/HyperCLOVA-X-SEED-Think-14B"

# 환경변수 설정
os.environ["LLM_MODEL"]    = HCX_MODEL
os.environ["LLM_API_BASE"] = "http://localhost:8000/v1"
os.environ["LLM_API_KEY"]  = "EMPTY"

print("📦 vLLM 설치 중...")
!pip install vllm bitsandbytes -q

print(f"\n🚀 vLLM API 서버 시작: {HCX_MODEL}")
print("   4-bit 양자화 (bitsandbytes) 적용 — ~8-10GB VRAM 사용")
print("   모델 로딩 약 5~15분 소요...\n")

# vLLM 백그라운드 실행
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", HCX_MODEL,
        "--host", "0.0.0.0",
        "--port", "8000",
        "--dtype", "bfloat16",
        "--quantization", "bitsandbytes",
        "--load-format", "bitsandbytes",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
        "--trust-remote-code",
        "--served-model-name", HCX_MODEL,
    ],
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)

# 서버 준비 대기 (최대 20분)
print("⏳ 서버 준비 대기 중 (최대 20분)", end="")
for _ in range(240):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print(f"\n✅ vLLM 서버 동작 중 — {HCX_MODEL}")
        print("   → 셀 4, 5를 실행한 뒤 셀 6 (앱 실행)으로 이동하세요")
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)
else:
    print("\n⚠️ 서버 시작 시간 초과. 로그 확인: !tail -50 /tmp/vllm.log")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 4/6  ▌ 프로젝트 코드 준비 (git clone / git pull)
#
# 처음 실행: 저장소 전체를 내려받습니다 (git clone)
# 재실행 시: 최신 코드로 업데이트합니다  (git pull)
# ══════════════════════════════════════════════════════════════════════

import os

REPO_URL = "https://github.com/machine-artisan/Hello-Clova-Agent.git"
REPO_DIR = "Hello-Clova-Agent"

if os.path.exists(REPO_DIR):
    print("🔄 최신 코드로 업데이트 중...")
    !git -C {REPO_DIR} pull
else:
    print("📂 프로젝트 다운로드 중...")
    !git clone {REPO_URL}

# 프로젝트 디렉토리로 이동
os.chdir(REPO_DIR)
print(f"\n✅ 현재 경로: {os.getcwd()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 5/6  ▌ 의존성 설치 + 환경변수 설정
#
# [개념] 환경변수(Environment Variable)란?
#   프로그램이 실행되는 환경에 주입하는 설정값입니다.
#   코드 안에 민감 정보(키, URL 등)를 하드코딩하지 않기 위해 사용합니다.
#
#   os.environ["KEY"] = "VALUE"  →  Python에서 환경변수 설정
# ══════════════════════════════════════════════════════════════════════

import os

print("📦 Python 패키지 설치 중... (약 1분 소요)")
!pip install -r requirements.txt -q

# Ollama OpenAI 호환 엔드포인트로 에이전트 연결
os.environ["LLM_API_BASE"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"]  = "EMPTY"   # Ollama는 키 불필요
# LLM_MODEL은 셀 3에서 이미 설정됨

print("\n✅ 설치 및 환경변수 설정 완료")
print(f"   LLM_API_BASE = {os.environ['LLM_API_BASE']}")
print(f"   LLM_MODEL    = {os.environ.get('LLM_MODEL', 'qwen2.5:7b')}")

## 🏗️ 시스템 아키텍처 — 에이전트가 동작하는 방식

```
┌─────────────────────────────────────────────────────────┐
│                   브라우저 (여러분의 PC)                 │
└───────────────────────┬─────────────────────────────────┘
                        │  HTTP (공개 URL)
┌───────────────────────▼─────────────────────────────────┐
│         🌐 웹서버  —  Gradio  (포트 7860)               │
│         ui/app.py  :  브라우저 ↔ 에이전트 연결          │
└───────────────────────┬─────────────────────────────────┘
                        │  Python 함수 호출
┌───────────────────────▼─────────────────────────────────┐
│         ⚙️  앱서버  —  LangGraph 파이프라인             │
│   Node1 input_parser → Node2 outline_generator          │
│   Node3 slide_writer → Node4 html_renderer              │
└───────────────────────┬─────────────────────────────────┘
                        │  HTTP (OpenAI API 형식)
┌───────────────────────▼─────────────────────────────────┐
│         🤖 API서버  —  Ollama  (포트 11434)             │
│         /v1/chat/completions  :  LLM 추론 제공          │
└───────────────────────┬─────────────────────────────────┘
                        │  GPU
┌───────────────────────▼─────────────────────────────────┐
│         🧠 LLM  —  Qwen 2.5 7B  (Colab GPU)            │
└─────────────────────────────────────────────────────────┘
```

| 계층 | 기술 | 역할 |
|------|------|------|
| **웹서버** | Gradio | 브라우저의 HTTP 요청 수신 + 응답 |
| **앱서버** | LangGraph | 비즈니스 로직 (에이전트 파이프라인 실행) |
| **API서버** | Ollama | LLM 추론을 REST API로 제공 |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 6/6  ▌ 에이전트 실행 🚀
#
# 셀 실행 후 출력되는 공개 URL을 브라우저에서 열어주세요.
#   예) Running on public URL: https://xxxx.gradio.live
#
# 중지하려면: 셀 왼쪽의 ■ 버튼 클릭
# ══════════════════════════════════════════════════════════════════════

print("🚀 Hello Clova Agent 시작 중...")
print("   아래에 공개 URL이 출력되면 브라우저에서 접속하세요 👇")
print()

!python ui/app.py

---

## 🔬 Phase 2 선택 실행 — 임베딩 모델 (RAG 준비)

아래 셀은 **Phase 2 RAG 연동**을 위한 임베딩 모델 설정입니다.  
Phase 1 체험만 원하면 실행하지 않아도 됩니다.

| 항목 | 내용 |
|------|------|
| 모델 | `BAAI/bge-m3` (다국어 임베딩, 한국어 우수) |
| 용도 | 외부 문서를 벡터로 변환하여 RAG 검색에 활용 |
| 크기 | ~570 MB |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# [선택] Phase 2  ▌ 임베딩 모델 준비 (RAG용)
#
# [개념] 임베딩(Embedding)이란?
#   텍스트를 의미를 담은 숫자 벡터로 변환하는 기술입니다.
#   비슷한 의미의 문장은 벡터 공간에서 가까운 위치에 배치되어
#   유사도 검색(RAG의 핵심)이 가능해집니다.
# ══════════════════════════════════════════════════════════════════════

import os

print("📦 임베딩 관련 패키지 설치 중...")
!pip install langchain-huggingface sentence-transformers -q

from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-m3"

# 모델 캐시 (재실행 시 다운로드 생략)
if not os.path.exists(EMBED_MODEL):
    print(f"📥 임베딩 모델 다운로드 중: {EMBED_MODEL}")
    model = SentenceTransformer(EMBED_MODEL)
    model.save(EMBED_MODEL)
    print("✅ 다운로드 완료 — 로컬 캐시에 저장됨")
else:
    print(f"✅ 캐시된 모델 사용: {EMBED_MODEL}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={
        "device": "cuda",
        "trust_remote_code": True,
    },
    encode_kwargs={"normalize_embeddings": True},
)

# 동작 테스트
test_vec = embeddings.embed_query("안녕하세요, 임베딩 테스트입니다.")
print(f"\n✅ 임베딩 모델 준비 완료")
print(f"   벡터 차원: {len(test_vec)}")
print(f"   샘플 벡터 (앞 5개): {[round(v, 4) for v in test_vec[:5]]}")

---

## 📚 더 알아보기

### 코드 구조 탐색

```
Hello-Clova-Agent/
├── agent/
│   ├── state.py              ← 파이프라인이 공유하는 데이터 구조
│   ├── llm.py                ← Ollama API 클라이언트
│   ├── graph.py              ← LangGraph 파이프라인 조립
│   └── nodes/
│       ├── input_parser.py   ← Node 1: 입력 분석
│       ├── outline_generator.py  ← Node 2: 목차 생성 (LLM)
│       ├── slide_writer.py   ← Node 3: 내용 작성 (LLM)
│       └── html_renderer.py  ← Node 4: HTML 변환
└── ui/app.py                 ← Gradio 웹 UI
```

### 참고 상용 서비스
- [Gamma](https://gamma.app) — AI 발표 자동 생성
- [SkyAI](https://skyai.io) — 한국어 특화 발표 생성
- [Manus](https://manus.im) — 멀티 에이전트 자동화

**이 노트북은 위 서비스의 핵심 파이프라인을 직접 구현한 학습용 클론입니다.**

---
*Hello Clova Agent · Phase 1 MVP · LangGraph + Ollama + Gradio + Reveal.js*